# simulate-fmri/ demo

An fMRI scan of a digital brain, acquired with the default ArbEPI protocol
(2.4 mm, 90 x 90 x 60, R = 10, 54 echoes per shot, volume TR 0.506 s, 60 s),
then reconstructed and scored against the truth.

1. **The sequence.** `sequences/ArbEPI.py` writes `scan_info.mat`: where in
   (ky, kz) and when every echo of every shot of every frame is acquired.
2. **The simulation.** [SNAKE-fMRI](https://github.com/mind-inria/snake-fmri)
   acquires a BrainWeb phantom, with a block-design BOLD activation in the
   occipital cortex, along exactly that schedule. T2* decays along each echo
   train and the BOLD signal changes between the shots of a frame.
3. **The reconstruction.** The output is a `<name>_preprocessed.h5` like the
   ones `preprocess/` writes from a real scan, so `recon/` reads it as is.
4. **The score.** The file also holds the true image, activated region and
   time course, in the layout `recon/testbed.py` scores against.

What the simulation leaves out (B0, ghosting, physiological noise, ...) is
listed in `README.md`. Kernel: `.venv-simulate` (see `README.md`'s setup).

In [ ]:
import importlib
import os
import sys
import time
import warnings
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd().parent if Path.cwd().name == "simulate-fmri" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))
# SNAKE's worker processes otherwise print one xsdata warning per MRD header they read
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.simplefilter("ignore")

from params import load_params  # noqa: E402
from recon import testbed  # noqa: E402
from recon.rss import run_rss  # noqa: E402
from recon.sense import main as recon_sense  # noqa: E402
from sample.gen_sampling_masks import resolve_omegas  # noqa: E402
from sequences.ArbEPI import generate_arbepi  # noqa: E402

# The folder name has a hyphen, so `import` cannot name it
simulate = importlib.import_module("simulate-fmri.simulate")

OUT = "output/simulate-fmri"
NAME = "demo"

In [ ]:
def show(volumes: dict, center=None, contour=None, cmaps=None, title=""):
    # Sagittal / coronal / axial slices through `center` (default: the middle), one row per volume
    fig, axes = plt.subplots(len(volumes), 3, figsize=(9, 2.7 * len(volumes)), squeeze=False)
    for i, (row, (name, vol)) in enumerate(zip(axes, volumes.items())):
        c = [n // 2 for n in vol.shape] if center is None else center
        vmax = np.percentile(vol, 99.8)
        cmap = (cmaps or {}).get(name, "gray")
        for j, (ax, view) in enumerate(zip(row, ["sagittal", "coronal", "axial"])):
            sl = [slice(None)] * 3
            sl[j] = c[j]
            im = ax.imshow(np.rot90(vol[tuple(sl)]), cmap=cmap, vmin=0, vmax=vmax)
            if contour is not None and contour[tuple(sl)].any():
                ax.contour(np.rot90(contour[tuple(sl)]), levels=[0.5], colors="c", linewidths=0.6)
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(view)
        row[0].set_ylabel(name)
        fig.colorbar(im, ax=row[2], fraction=0.046)
    fig.suptitle(title)
    plt.show()

## 1. The sequence

Only `generate_arbepi` is needed (not the calibration, noise and deGRE
sequences): it partitions each frame's sampling mask into echo trains and
saves the result in `scan_info.mat`. To simulate a different protocol, edit
`params.py`; to simulate a past session, point at its `scan_info.mat`.

In [ ]:
params = load_params(output_dir=OUT)
generate_arbepi(resolve_omegas(params), params)

protocol = simulate.load_protocol(f"{OUT}/scan_info.mat")
print(f"\n{protocol.n_frames} frames x {protocol.n_shots} shots x {protocol.etl} echoes, "
      f"R = {protocol.acceleration:.1f}, {params.sampling_method} sampling, "
      f"{params.epi_trajectory} ordering")
print(f"shot TR {protocol.tr_shot_ms:.1f} ms, TE {protocol.te_ms:.1f} ms, "
      f"echo spacing {1e3 * protocol.echo_spacing_ms:.0f} us, flip {protocol.fa_deg:.1f} deg")

## 2. The simulation

Defaults: BrainWeb subject 4 (white matter, gray matter, CSF at 3 T), 16
receive coils in two rings around z, T2* decay along each echo train, a 10 s
on / 10 s off block design peaking at TE / 1000 ms = 3% in gray matter, and
white k-space noise at SNAKE's `snr = 1000`.

One SNAKE repetition is one ArbEPI shot: the phantom is updated, excited and
sampled once per echo train.

In [ ]:
t0 = time.time()
paths = simulate.simulate(f"{OUT}/scan_info.mat", OUT, NAME, n_workers=16)
print(f"{time.time() - t0:.0f} s")
FN = paths["preprocessed"]

The output has the datasets `recon/` reads from a preprocessed scan, plus the
`truth` group.

In [ ]:
with h5py.File(FN) as f:
    f.visititems(lambda name, obj: print(f"{name:22s} {obj.shape} {obj.dtype}")
                 if isinstance(obj, h5py.Dataset) else None)
    print({k: (round(float(v), 6) if np.isscalar(v) and not isinstance(v, (str, bytes)) else v)
           for k, v in f.attrs.items() if k not in ("fov",)})
    omegas, echo_times = f["omegas"][()], f["echo_times"][()]
    smaps = f["smaps"][()]
    x0, amp_map = f["truth/x0"][()], f["truth/amp_map"][()]
    roi, wave = f["truth/roi_masks"][0], f["truth/waveforms"][0]
    stimulus = f["truth/stimulus_shots"][()]
    amp, volume_tr, tr_shot = f["truth"].attrs["amp"], f.attrs["volume_tr"], f.attrs["tr_shot"]
nt = len(wave)

fig, ax = plt.subplots(1, 4, figsize=(15, 3.2), width_ratios=[1, 1.15, 1, 1.6])
ax[0].imshow(omegas[:, :, 0].T, origin="lower", cmap="gray")
ax[0].set_title(f"(ky, kz) mask, frame 0 ({omegas[:, :, 0].sum()} of {omegas[:, :, 0].size})")
t_ms = np.where(omegas[:, :, 0], 1e3 * echo_times[:, :, 0], np.nan)
im = ax[1].imshow(t_ms.T, origin="lower")
ax[1].set_title("echo time (ms)")
plt.colorbar(im, ax=ax[1])
ax[2].imshow(np.abs(smaps[:, :, smaps.shape[2] // 2, 0]).T, origin="lower", cmap="viridis")
ax[2].set_title("|sensitivity| of coil 0, axial")
t = (np.arange(nt) + 0.5) * volume_tr
ax[3].fill_between(np.arange(len(stimulus)) * tr_shot, 0, stimulus, color="0.85", label="stimulus")
ax[3].plot(t, wave, label="activation w(t)")
ax[3].set_xlabel("s")
ax[3].set_title("time course of the activated region")
ax[3].legend(loc="lower right", fontsize=8)
for a in ax[:3]:
    a.set_xticks([])
    a.set_yticks([])
plt.tight_layout()
plt.show()

The truth: `x0`, the noise-free image at TE, and `amp_map`, each voxel's peak
fractional signal change. The activation is gray matter inside an ellipsoid at
the occipital pole, so it follows the cortex; the contour is `roi_masks`, the
voxels that are nearly all tissue and at half the largest change or more. Frame t of the true series is
`x0 * (1 + amp_map * w(t))`.

In [ ]:
center = np.round(np.argwhere(roi).mean(axis=0)).astype(int)
show({"x0": x0, "amp_map": amp_map}, center=center, contour=roi, cmaps={"amp_map": "magma"},
     title=f"truth, through the activated region ({roi.sum()} voxels, median change {100 * amp:.2f}%)")

## 3. Reconstruction

`recon/` needs nothing but the file. Root-sum-of-squares of the zero-filled
k-space first: at R = 10 it shows the anatomy through the aliasing.

In [ ]:
img_rss = run_rss(FN)
show({"RSS, frame 0": img_rss[..., 0], "RSS, mean": img_rss.mean(axis=-1)}, center=center)

Then iterative SENSE with the L1-wavelet + TV regularizer and the temporal
high-pass penalty (`--reg wavelet-tv --hp-weight 3`), all frames jointly.
`sigma1A=1` skips the power iteration: with unit root-sum-of-squares
sensitivities and no B0 model the operator's norm is 1.

In [ ]:
t0 = time.time()
out = recon_sense(OUT, NAME, "wavelet-tv", hp_weight=3.0, sigma1A=1.0, tag="hp3")
print(f"{time.time() - t0:.0f} s")
FN_RECON = out + ".h5"

## 4. Score

`recon.testbed.score` compares magnitudes after one global scale: the error of
the mean image and of each frame, the temporal fluctuation of voxels that
should be constant, and for the activated region the recovered amplitude
(1 = exact), its t-score, and the leakage into a surrounding shell.

In [ ]:
score = testbed.score(FN, FN_RECON)
for k, v in score.items():
    print(f"{k:34s} {v:8.3f}")

In [ ]:
testbed.panel(FN, FN_RECON, f"{OUT}/{NAME}_panel.png", "wavelet-TV + temporal high-pass, R = 10")
from IPython.display import Image  # noqa: E402

Image(f"{OUT}/{NAME}_panel.png", width=700)

The time course of the activated region, reconstructed against true. Both are
means over `roi_masks` of the signal relative to `x0`.

In [ ]:
with h5py.File(FN_RECON) as f:
    X = np.abs(f["X_recon"][()])
truth_roi = 1 + (amp_map[roi][:, None] * wave[None, :]).mean(axis=0)
recon_roi = (score["alpha"] * X[roi] / x0[roi][:, None]).mean(axis=0)

fig, ax = plt.subplots(figsize=(9, 3))
ax.fill_between(np.arange(len(stimulus)) * tr_shot, 0.9, 0.9 + 0.2 * stimulus, color="0.9")
ax.plot(t, 100 * (truth_roi - 1), "k", label="truth")
ax.plot(t, 100 * (recon_roi - recon_roi.mean()), label="reconstruction (mean removed)")
ax.set_ylim(-3, 3)
ax.set_xlabel("s")
ax.set_ylabel("signal change (%)")
ax.legend()
plt.show()

## Where to go from here

- **Another protocol**: change `params.py` (`R`, `ETL`, `sampling_method`,
  `epi_trajectory`, resolution, `duration`) and rerun from the top. The
  simulation reads everything from `scan_info.mat`.
- **Isolate an effect**: `model="simple"` removes the T2* decay along the echo
  train (the reconstruction's forward model is then exact), `snr=np.inf` the
  noise, `handlers=[]` the activation.
- **Another experiment**: `block_on`, `block_off`, `delta_r2s` and `roi`, or
  any list of SNAKE handlers through `handlers=`.
- **From the command line**: `python -m simulate-fmri.simulate`,
  `python -m recon.sense`, `python -m recon.testbed score` (see `README.md`).